In [1]:
import os
import pandas as pd
from pathlib import Path

os.chdir ('C:\\Users\\hamod4691\\Code\\arctic-benthic-biodiversity\\OBIS')

# Fix dates

In [2]:
merged_OBIS = pd.read_parquet("merged_OBIS.parquet")

merged_OBIS["startDayOfYear"] = pd.to_datetime(
    merged_OBIS["date_start"], unit="ms", utc=True, errors="coerce"
).dt.dayofyear.astype("Int64")
merged_OBIS["endDayOfYear"] = pd.to_datetime(
    merged_OBIS["date_end"], unit="ms", utc=True, errors="coerce"
).dt.dayofyear.astype("Int64")

merged_OBIS.to_parquet("merged_OBIS.parquet", index=False)

# Check column coverage

In [3]:
column_coverage_OBIS = pd.DataFrame({
    "column": merged_OBIS.columns,
    "non_null_count": merged_OBIS.notna().sum().values,
    "total_rows": len(merged_OBIS),
})

column_coverage_OBIS["coverage_pct"] = (
    column_coverage_OBIS["non_null_count"] / column_coverage_OBIS["total_rows"] * 100
)

column_coverage_OBIS = column_coverage_OBIS.sort_values(
    by=["coverage_pct", "non_null_count"], ascending=False
)

column_coverage_OBIS.to_csv("column_coverage_OBIS.csv", index=False)

column_coverage_OBIS.head()

,column,non_null_count,total_rows,coverage_pct
0,basisOfRecord,1533558,1533558,100.0
10,decimalLatitude,1533558,1533558,100.0
11,decimalLongitude,1533558,1533558,100.0
27,kingdom,1533558,1533558,100.0
28,kingdomid,1533558,1533558,100.0


# Prepare for merging

In [4]:
merged_OBIS = pd.read_parquet("merged_OBIS.parquet")

dtype_spec = pd.read_excel(Path("..") / "column_data_types.xlsx", sheet_name=0)
dtype_map = dict(zip(dtype_spec["column"], dtype_spec["final"]))

column_map = {
    "sourceID": "id",
    "verbatimID": "occurrenceID",
    "datasetKey": "dataset_id",
    "datasetTitle": "datasetName",
    "decimalLatitudeStart": "decimalLatitude",
    "decimalLongitudeStart": "decimalLongitude",
    "footprintWKT": "footprintWKT",
    "coordinateUncertaintyInMeters": "coordinateUncertaintyInMeters",
    "depthA_m": "maximumDepthInMeters",
    "depthB_m": "minimumDepthInMeters",
    "eventID": "eventID",
    "parentEventID": "parentEventID",
    "waterBody": "waterBody",
    "locationID": "locationID",
    "verbatimDate": "eventDate",
    "startDayOfYear": "startDayOfYear",
    "endDayOfYear": "endDayOfYear",
    "year": "date_year",
    "eventRemarks": "eventRemarks",
    "verbatimScientificName": "originalScientificName",
    "acceptedScientificName": "scientificName",
    "identificationQualifier": "identificationQualifier",
    "lifeStage": "lifeStage",
    "sex": "sex",
    "taxonomicStatus": "taxonomicStatus",
    "taxonRank": "taxonRank",
    "phylum": "phylum",
    "class": "class",
    "order": "order",
    "family": "family",
    "genus": "genus",
    "specificEpithet": "specificEpithet",
    "validAuthority": "scientificNameAuthorship",
    "aphiaID": "aphiaID",
    "individualCount": "individualCount",
    "dynamicProperties": "dynamicProperties",
    "basisOfRecord": "basisOfRecord",
    "samplingEffort": "samplingEffort",
    "sampleSizeValue": "sampleSizeValue",
    "sampleSizeUnit": "sampleSizeUnit",
}

output_columns = [
    "ID",
    "sourceID",
    "verbatimID",
    "sourceDatabase",
    "datasetKey",
    "datasetTitle",
    "issue",
    "decimalLatitudeStart",
    "decimalLongitudeStart",
    "decimalLatitudeStop",
    "decimalLongitudeStop",
    "footprintWKT",
    "coordinateUncertaintyInMeters",
    "hasGeospatialIssues",
    "depthA_m",
    "depthAType",
    "depthB_m",
    "depthBType",
    "gebcoDepth",
    "eventID",
    "parentEventID",
    "waterBody",
    "largeMarineEcosystem",
    "locationID",
    "verbatimDate",
    "startDayOfYear",
    "endDayOfYear",
    "year",
    "eventRemarks",
    "verbatimScientificName",
    "acceptedScientificName",
    "identificationQualifier",
    "lifeStage",
    "sex",
    "taxonomicStatus",
    "taxonomicIssue",
    "taxonRank",
    "phylum",
    "class",
    "order",
    "family",
    "genus",
    "specificEpithet",
    "validAuthority",
    "aphiaID",
    "individualCount",
    "areaSampled",
    "areaSampledUnit",
    "samplingEffort",
    "percentCoverage",
    "biomass",
    "biomassUnit",
    "dynamicProperties",
    "basisOfRecord",
    "occurrenceRemarks",
    "gear",
    "gearCategory",
    "meshSize_mm",
    "sieveFraction_µm",
    "flagged",
    "flaggedComment", 
    "acceptedTaxonKey"
]


def normalize_dtype(dtype_name):
    dtype_name = str(dtype_name).strip().lower()
    return {
        "int32": "Int32",
        "float32": "Float32",
        "float64": "Float64",
        "boolean": "boolean",
        "string": "string",
    }.get(dtype_name, dtype_name)


def empty_series(index, dtype_name):
    dtype_name = normalize_dtype(dtype_name)
    if dtype_name in {"Int32", "Float32", "Float64", "boolean", "string"}:
        return pd.Series(pd.NA, index=index, dtype=dtype_name)
    return pd.Series(pd.NA, index=index, dtype="string")


def coerce_series(series, dtype_name):
    dtype_name = normalize_dtype(dtype_name)
    if dtype_name == "boolean":
        return series.astype("boolean")
    if dtype_name in {"Int32", "Float32", "Float64"}:
        return pd.to_numeric(series, errors="coerce").astype(dtype_name)
    return series.astype("string")


obis_final = pd.DataFrame(index=merged_OBIS.index)
missing_sources = []

for col in output_columns:
    target_dtype = dtype_map.get(col, "string")
    if col in column_map:
        src = column_map[col]
        if src in merged_OBIS.columns:
            obis_final[col] = coerce_series(merged_OBIS[src], target_dtype)
        else:
            obis_final[col] = empty_series(merged_OBIS.index, target_dtype)
            missing_sources.append(src)
    else:
        obis_final[col] = empty_series(merged_OBIS.index, target_dtype)

obis_final = obis_final[output_columns]
obis_final.to_parquet("obis_final.parquet", index=False)

print(f"Saved obis_final.parquet with {len(obis_final):,} rows and {len(obis_final.columns)} columns.")
if missing_sources:
    print("Missing source columns in merged_OBIS:", sorted(set(missing_sources)))

Saved obis_final.parquet with 1,533,558 rows and 62 columns.


In [5]:
obis_final = pd.read_parquet("obis_final.parquet")
obis_final["sourceDatabase"] = "OBIS"
obis_final.to_parquet("obis_final.parquet", index=False)
print("Added sourceDatabase column with value 'OBIS'. Saved obis_final.parquet.")

Added sourceDatabase column with value 'OBIS'. Saved obis_final.parquet.


In [6]:
obis_final.loc[obis_final["depthA_m"].notna(), "depthAType"] = "maximum"
obis_final.loc[obis_final["depthB_m"].notna(), "depthBType"] = "minimum"

obis_final.to_parquet("obis_final.parquet", index=False)
obis_final[["depthA_m", "depthAType", "depthB_m", "depthBType"]].head()
obis_final.to_parquet("obis_final.parquet", index=False)
print("Saved obis_final.parquet.")

Saved obis_final.parquet.


# Check data types

In [7]:
obis_final = pd.read_parquet("obis_final.parquet")

# Column type overview
column_types = pd.DataFrame({
    "column": obis_final.columns,
    "dtype": obis_final.dtypes.astype(str).values,
    "non_null_count": obis_final.notna().sum().values,
    "null_count": obis_final.isna().sum().values,
    "non_null_pct": (obis_final.notna().sum().values / len(obis_final)) * 100,
})

column_types = column_types.sort_values(
    by=["non_null_count", "column"], ascending=[False, True]
 )

column_types.to_csv("obis_final_column_types.csv", index=False)

column_types.head(20)

,column,dtype,non_null_count,null_count,non_null_pct
30,acceptedScientificName,string,1533558,0,100.000000
44,aphiaID,Int32,1533558,0,100.000000
53,basisOfRecord,string,1533558,0,100.000000
4,datasetKey,string,1533558,0,100.000000
7,decimalLatitudeStart,Float64,1533558,0,100.000000
8,decimalLongitudeStart,Float64,1533558,0,100.000000
37,phylum,string,1533558,0,100.000000
3,sourceDatabase,str,1533558,0,100.000000
1,sourceID,string,1533558,0,100.000000
29,verbatimScientificName,string,1532890,668,99.956441
